# 🕵️ Data Detective Agent
> Upload any messy CSV — the agent **auto-cleans it**, **detects anomalies**, **generates charts**, and writes a full **PDF insight report**. Powered by DeepSeek.

---
**What the agent does autonomously:**
1. 🔍 Profiles your data (shape, types, nulls, duplicates)
2. 🧹 Cleans it (fixes nulls, removes duplicates, flags outliers)
3. 📊 Generates 4 charts saved as PNG files
4. 🧠 Asks DeepSeek to interpret the findings and write insights
5. 📄 Exports everything into a professional PDF report

## ⚙️ Cell 1 — Install & Import

In [ ]:
!pip install openai pandas numpy matplotlib seaborn scipy fpdf2 --quiet

In [ ]:
import json, os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from openai import OpenAI
from fpdf import FPDF
from datetime import datetime
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('✅ All imports successful!')

## 🔑 Cell 2 — DeepSeek Configuration

In [ ]:
import os, getpass
DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY") or getpass.getpass("DeepSeek API key: ")

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)
MODEL = "deepseek-chat"
print(f'✅ DeepSeek ready | Model: {MODEL}')

## 📁 Cell 3 — Load Your CSV
> **Change `CSV_PATH` to your file path, or use the sample generator below.**

In [ ]:
# ── Option A: use your own CSV ─────────────────────────────────────────────
CSV_PATH = "sample_dirty_data.csv"   # ← change this to your file

# ── Option B: generate a sample dirty dataset ──────────────────────────────
def generate_sample_csv(path="sample_dirty_data.csv"):
    np.random.seed(42)
    n = 200
    df = pd.DataFrame({
        'age':              np.random.normal(35, 10, n).astype(int),
        'salary':           np.random.normal(55000, 15000, n),
        'department':       np.random.choice(['HR','Engineering','Sales','Marketing', None], n),
        'performance_score':np.random.choice([1, 2, 3, 4, 5, None], n),
        'years_at_company': np.random.normal(5, 3, n),
        'bonus':            np.random.normal(3000, 1500, n),
    })
    df.loc[np.random.choice(n, 20), 'salary']           = np.nan
    df.loc[np.random.choice(n, 10), 'age']              = np.random.choice([-5, 999, 0], 10)
    df.loc[np.random.choice(n, 5),  'salary']           = np.random.choice([500000, -1000], 5)
    df.loc[np.random.choice(n, 15), 'years_at_company'] = np.nan
    df = pd.concat([df, df.iloc[:8]], ignore_index=True)  # add duplicates
    df.to_csv(path, index=False)
    print(f'✅ Sample dirty CSV created: {path} ({df.shape[0]} rows × {df.shape[1]} cols)')
    return path

# Uncomment the line below to generate the sample file:
# CSV_PATH = generate_sample_csv()

df_raw = pd.read_csv(CSV_PATH)
print(f'📂 Loaded: {CSV_PATH}')
print(f'   Shape : {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
df_raw.head()

## 🤖 Cell 4 — Agent Tools

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "profile_data",
            "description": "Profile the raw dataset: shape, dtypes, null counts, duplicate count, numeric summary stats.",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "clean_data",
            "description": "Clean the dataset: drop duplicates, fill nulls with median/mode, flag outliers using IQR.",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "detect_anomalies",
            "description": "Detect anomalies using Z-score and IQR methods on all numeric columns.",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_charts",
            "description": "Generate and save 4 charts: distributions, correlation heatmap, outlier boxplots, category breakdown.",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_insights",
            "description": "Ask DeepSeek to interpret the findings and write a professional data insights narrative.",
            "parameters": {
                "type": "object",
                "properties": {
                    "summary_for_llm": {
                        "type": "string",
                        "description": "A concise text summary of all findings to send to the LLM for interpretation."
                    }
                },
                "required": ["summary_for_llm"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "export_pdf_report",
            "description": "Export a full PDF report with all findings, stats, anomalies, charts, and AI insights.",
            "parameters": {
                "type": "object",
                "properties": {
                    "report_title": {
                        "type": "string",
                        "description": "Title for the PDF report."
                    }
                },
                "required": ["report_title"]
            }
        }
    }
]

print('✅ Tools defined:', [t['function']['name'] for t in TOOLS])

## 🛠️ Cell 5 — Tool Executor (Real Actions)

In [ ]:
# ── Shared agent state ─────────────────────────────────────────────────────
state = {
    "df_raw":      None,
    "df_clean":    None,
    "profile":     {},
    "anomalies":   {},
    "chart_paths": [],
    "insights":    "",
    "pdf_path":    ""
}

# ─────────────────────────────────────────────────────────────────────────────
def tool_profile_data():
    df = state["df_raw"]
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()

    profile = {
        "rows": len(df), "cols": len(df.columns),
        "numeric_cols": num_cols, "categorical_cols": cat_cols,
        "null_counts": df.isnull().sum().to_dict(),
        "null_pct": (df.isnull().mean() * 100).round(1).to_dict(),
        "duplicates": int(df.duplicated().sum()),
        "stats": df[num_cols].describe().round(2).to_dict() if num_cols else {}
    }
    state["profile"] = profile

    print("\n🔍 DATA PROFILE")
    print(f"{'═'*55}")
    print(f"  Rows: {profile['rows']}  |  Columns: {profile['cols']}")
    print(f"  Numeric : {num_cols}")
    print(f"  Categorical: {cat_cols}")
    print(f"  Duplicates: {profile['duplicates']}")
    print("\n  Null counts:")
    for col, cnt in profile['null_counts'].items():
        if cnt > 0:
            print(f"    {col}: {cnt} ({profile['null_pct'][col]}%)")
    return json.dumps({"status": "ok", "rows": profile['rows'], "nulls_found": sum(v > 0 for v in profile['null_counts'].values())})

# ─────────────────────────────────────────────────────────────────────────────
def tool_clean_data():
    df = state["df_raw"].copy()
    before = len(df)

    # 1. Drop duplicates
    df = df.drop_duplicates()
    dupes_removed = before - len(df)

    # 2. Fill numeric nulls with median
    num_cols = df.select_dtypes(include=np.number).columns
    fills = {}
    for col in num_cols:
        if df[col].isnull().any():
            med = df[col].median()
            df[col] = df[col].fillna(med)
            fills[col] = round(med, 2)

    # 3. Fill categorical nulls with mode
    cat_cols = df.select_dtypes(include='object').columns
    for col in cat_cols:
        if df[col].isnull().any():
            mode = df[col].mode()[0]
            df[col] = df[col].fillna(mode)
            fills[col] = mode

    # 4. Flag outliers using IQR (add column, don't remove)
    outlier_flags = pd.Series(False, index=df.index)
    for col in num_cols:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        outlier_flags |= ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR))
    df['_is_outlier'] = outlier_flags
    outlier_count = int(outlier_flags.sum())

    state["df_clean"] = df

    print("\n🧹 DATA CLEANING COMPLETE")
    print(f"{'═'*55}")
    print(f"  Duplicates removed : {dupes_removed}")
    print(f"  Nulls filled       : {fills}")
    print(f"  Outlier rows flagged: {outlier_count}")
    print(f"  Clean shape        : {df.shape}")
    return json.dumps({"status": "ok", "dupes_removed": dupes_removed, "outliers_flagged": outlier_count})

# ─────────────────────────────────────────────────────────────────────────────
def tool_detect_anomalies():
    df = state["df_clean"]
    num_cols = [c for c in df.select_dtypes(include=np.number).columns if c != '_is_outlier']
    anomalies = {}

    for col in num_cols:
        z = np.abs(stats.zscore(df[col].dropna()))
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        iqr_out = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
        anomalies[col] = {
            "zscore_outliers": int((z > 3).sum()),
            "iqr_outliers":    len(iqr_out),
            "min": round(float(df[col].min()), 2),
            "max": round(float(df[col].max()), 2),
            "mean": round(float(df[col].mean()), 2)
        }

    state["anomalies"] = anomalies

    print("\n🚨 ANOMALY DETECTION")
    print(f"{'═'*55}")
    for col, info in anomalies.items():
        flag = " ⚠️" if info['iqr_outliers'] > 0 else ""
        print(f"  {col}{flag}")
        print(f"    Range: [{info['min']} → {info['max']}]  |  Mean: {info['mean']}")
        print(f"    Z-score outliers: {info['zscore_outliers']}  |  IQR outliers: {info['iqr_outliers']}")
    return json.dumps({"status": "ok", "columns_analyzed": len(anomalies)})

# ─────────────────────────────────────────────────────────────────────────────
def tool_generate_charts():
    df = state["df_clean"]
    num_cols = [c for c in df.select_dtypes(include=np.number).columns if c != '_is_outlier']
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    paths = []

    # Chart 1 — Distributions
    fig, axes = plt.subplots(1, len(num_cols), figsize=(4*len(num_cols), 4))
    if len(num_cols) == 1: axes = [axes]
    for ax, col in zip(axes, num_cols):
        sns.histplot(df[col], ax=ax, kde=True, color='steelblue', alpha=0.7)
        ax.set_title(f'{col}', fontsize=11, fontweight='bold')
        ax.set_xlabel('')
    fig.suptitle('📊 Feature Distributions', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    p1 = 'chart_distributions.png'
    plt.savefig(p1, bbox_inches='tight'); plt.close()
    paths.append(p1)

    # Chart 2 — Correlation heatmap
    if len(num_cols) > 1:
        fig, ax = plt.subplots(figsize=(7, 5))
        corr = df[num_cols].corr()
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                    center=0, ax=ax, linewidths=0.5, square=True)
        ax.set_title('🔗 Correlation Heatmap', fontsize=13, fontweight='bold')
        plt.tight_layout()
        p2 = 'chart_correlation.png'
        plt.savefig(p2, bbox_inches='tight'); plt.close()
        paths.append(p2)

    # Chart 3 — Boxplots (outlier detection)
    fig, axes = plt.subplots(1, len(num_cols), figsize=(4*len(num_cols), 4))
    if len(num_cols) == 1: axes = [axes]
    for ax, col in zip(axes, num_cols):
        bp = ax.boxplot(df[col].dropna(), patch_artist=True,
                        boxprops=dict(facecolor='lightcoral', alpha=0.6),
                        medianprops=dict(color='darkred', linewidth=2))
        ax.set_title(col, fontsize=11, fontweight='bold')
        ax.set_xticks([])
    fig.suptitle('📦 Outlier Boxplots', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    p3 = 'chart_boxplots.png'
    plt.savefig(p3, bbox_inches='tight'); plt.close()
    paths.append(p3)

    # Chart 4 — Category breakdown
    if cat_cols:
        col = cat_cols[0]
        vc = df[col].value_counts()
        fig, ax = plt.subplots(figsize=(7, 4))
        bars = ax.bar(vc.index, vc.values, color=sns.color_palette('muted', len(vc)), edgecolor='white')
        ax.set_title(f'🗂️ Category Distribution: {col}', fontsize=13, fontweight='bold')
        ax.set_xlabel(col); ax.set_ylabel('Count')
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
        plt.tight_layout()
        p4 = 'chart_categories.png'
        plt.savefig(p4, bbox_inches='tight'); plt.close()
        paths.append(p4)

    state["chart_paths"] = paths
    print(f"\n📊 CHARTS GENERATED")
    print(f"{'═'*55}")
    for p in paths:
        print(f"   ✅ Saved: {p}")
    return json.dumps({"status": "ok", "charts": paths})

# ─────────────────────────────────────────────────────────────────────────────
def tool_write_insights(summary_for_llm: str):
    print("\n🧠 REQUESTING AI INSIGHTS FROM DEEPSEEK...")
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=1000,
        messages=[
            {"role": "system", "content": (
                "You are a senior data scientist writing an executive summary of a data quality report. "
                "Be professional, specific, and actionable. Use short paragraphs. No bullet lists."
            )},
            {"role": "user", "content": f"Here are the findings from our data analysis:\n\n{summary_for_llm}\n\nWrite a professional 3-paragraph insight report: 1) Data quality overview, 2) Key anomalies and risks, 3) Recommended actions."}
        ]
    )
    insights = response.choices[0].message.content.strip()
    state["insights"] = insights
    print("\n💬 AI INSIGHTS:")
    print(f"{'─'*55}")
    print(insights)
    return json.dumps({"status": "ok", "insight_length": len(insights)})

# ─────────────────────────────────────────────────────────────────────────────
def tool_export_pdf_report(report_title: str):
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    # Title
    pdf.set_font('Helvetica', 'B', 20)
    pdf.set_fill_color(30, 80, 162)
    pdf.set_text_color(255, 255, 255)
    pdf.cell(0, 14, f'  {report_title}', fill=True, ln=True)
    pdf.ln(2)
    pdf.set_font('Helvetica', '', 9)
    pdf.set_text_color(120, 120, 120)
    pdf.cell(0, 6, f'Generated by Data Detective Agent | {datetime.now().strftime("%Y-%m-%d %H:%M")}', ln=True)
    pdf.ln(4)

    def section(title):
        pdf.set_font('Helvetica', 'B', 13)
        pdf.set_text_color(30, 80, 162)
        pdf.cell(0, 8, title, ln=True)
        pdf.set_draw_color(30, 80, 162)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(3)
        pdf.set_text_color(40, 40, 40)

    def body(text):
        pdf.set_font('Helvetica', '', 10)
        pdf.multi_cell(0, 6, str(text))
        pdf.ln(2)

    # 1. Profile
    p = state["profile"]
    section('1. Data Profile')
    body(f"File: {CSV_PATH}")
    body(f"Raw shape: {p['rows']} rows × {p['cols']} columns")
    body(f"Numeric columns : {', '.join(p['numeric_cols'])}")
    body(f"Categorical columns: {', '.join(p['categorical_cols'])}")
    body(f"Duplicate rows: {p['duplicates']}")
    null_lines = [f"{c}: {v} nulls ({p['null_pct'][c]}%)" for c, v in p['null_counts'].items() if v > 0]
    body("Missing values:\n" + "\n".join(null_lines) if null_lines else "No missing values found.")

    # 2. Cleaning
    section('2. Cleaning Summary')
    df_c = state["df_clean"]
    body(f"Clean shape: {df_c.shape[0]} rows × {df_c.shape[1]} columns")
    body(f"Outlier rows flagged: {int(df_c['_is_outlier'].sum())}")

    # 3. Anomalies
    section('3. Anomaly Detection')
    for col, info in state["anomalies"].items():
        body(f"{col}:\n  Range [{info['min']} → {info['max']}] | Mean {info['mean']} | Z-score outliers: {info['zscore_outliers']} | IQR outliers: {info['iqr_outliers']}")

    # 4. AI Insights
    section('4. AI-Generated Insights')
    body(state["insights"] if state["insights"] else "(no insights generated)")

    # 5. Charts
    section('5. Visualizations')
    for chart_path in state["chart_paths"]:
        if os.path.exists(chart_path):
            pdf.add_page()
            label = chart_path.replace('chart_','').replace('.png','').replace('_',' ').title()
            pdf.set_font('Helvetica', 'B', 11)
            pdf.set_text_color(40, 40, 40)
            pdf.cell(0, 8, label, ln=True)
            pdf.image(chart_path, x=10, w=190)

    fname = report_title.lower().replace(' ', '_').replace('/', '-') + '_report.pdf'
    pdf.output(fname)
    state["pdf_path"] = fname

    print(f"\n📄 PDF REPORT EXPORTED")
    print(f"{'═'*55}")
    print(f"   File : {fname}")
    print(f"   Pages: {pdf.page}")
    print(f"   Charts embedded: {len(state['chart_paths'])}")
    return json.dumps({"status": "ok", "path": fname})

# ─────────────────────────────────────────────────────────────────────────────
def run_tool(name, args):
    dispatch = {
        "profile_data":     lambda: tool_profile_data(),
        "clean_data":       lambda: tool_clean_data(),
        "detect_anomalies": lambda: tool_detect_anomalies(),
        "generate_charts":  lambda: tool_generate_charts(),
        "write_insights":   lambda: tool_write_insights(args.get("summary_for_llm", "")),
        "export_pdf_report":lambda: tool_export_pdf_report(args.get("report_title", "Data Report")),
    }
    fn = dispatch.get(name)
    if fn: return fn()
    return json.dumps({"status": "error", "message": f"Unknown tool: {name}"})

print('✅ Tool executor ready')

## 🚀 Cell 6 — The Agent Loop

In [ ]:
def run_agent(csv_path: str):
    # Reset state
    state.update({"df_raw": None, "df_clean": None, "profile": {},
                  "anomalies": {}, "chart_paths": [], "insights": "", "pdf_path": ""})
    state["df_raw"] = pd.read_csv(csv_path)

    system_prompt = """You are the Data Detective Agent. You receive a dataset and MUST call these tools IN ORDER:
1. profile_data    → understand the raw data
2. clean_data      → fix nulls, duplicates, flag outliers
3. detect_anomalies → find statistical anomalies
4. generate_charts  → visualize everything
5. write_insights   → summarize your full findings as summary_for_llm, then call this tool
6. export_pdf_report → export the final report with a descriptive title

Always call ALL 6 tools. Never skip any."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": f"Analyze this dataset: '{csv_path}'. Run the full Data Detective pipeline."}
    ]

    print(f"\n🕵️  DATA DETECTIVE AGENT STARTED")
    print(f"📂 File: {csv_path}")
    print(f"{'═'*55}")

    for step in range(1, 12):  # max 12 steps
        print(f"\n⚙️  Step {step}: calling DeepSeek...")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto"
        )

        msg = response.choices[0].message

        if not msg.tool_calls:
            if msg.content:
                print(f"\n💬 Agent: {msg.content}")
            print(f"\n{'═'*55}")
            print("✅ Agent finished!")
            break

        messages.append({"role": "assistant", "content": msg.content, "tool_calls": [
            {"id": tc.id, "type": "function",
             "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
            for tc in msg.tool_calls
        ]})

        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            print(f"   🔧 Tool: `{name}`")
            result = run_tool(name, args)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    print(f"\n🎉 DONE! Report saved to: {state['pdf_path']}")

print('✅ Agent ready — run Cell 7!')

## ▶️ Cell 7 — Run It!
> Change `CSV_PATH` to your own file, or run `generate_sample_csv()` from Cell 3 first.

In [ ]:
run_agent(CSV_PATH)

## 🖼️ Cell 8 — Preview Charts Inline

In [ ]:
from IPython.display import display, Image
for path in state['chart_paths']:
    if os.path.exists(path):
        print(f"\n📊 {path}")
        display(Image(filename=path, width=700))

## 🔄 Cell 9 — Try Your Own CSV

In [ ]:
# Drop your own CSV here!
my_csv = "your_data.csv"   # ← change this
run_agent(my_csv)